# ECG arrhythmia: exploratory data analysis
Run `python -m src.data` first, so that `data/processed/beats.npz` exists.

Questions:
1. How imbalanced are the classes, and does the imbalance differ between train and test patients?
2. What does an average beat of each class look like?
3. How well does timing (RR intervals) alone separate the classes?
4. How different are patients from each other? (This is why a beat-level split leaks.)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.data import load_dataset
from src.split import patient_split

X, rr, y, patient = load_dataset(ROOT / "data" / "processed" / "beats.npz")
masks = patient_split(patient)
df = pd.DataFrame({"label": [config.CLASSES[i] for i in y], "patient": patient,
                   "pre_rr": rr[:, 0], "post_rr": rr[:, 1], "pre_rr_ratio": rr[:, 2]})
df["split"] = np.select([masks["train"], masks["val"], masks["test"]], ["train", "val", "test"], default="other")
print(f"{len(df):,} beats from {df.patient.nunique()} patients")

## 1. Class balance per split

In [ ]:
counts = pd.crosstab(df.split, df.label)[config.CLASSES]
display(counts)
display((counts.div(counts.sum(axis=1), axis=0) * 100).round(2).rename(columns=lambda c: c + " %"))

## 2. Average beat shape per class (mean ± 1 std)

In [ ]:
t = (np.arange(config.BEAT_LEN) - config.WINDOW_BEFORE) / config.FS * 1000
fig, axes = plt.subplots(1, 5, figsize=(18, 3.2), sharey=True)
for i, (ax, c) in enumerate(zip(axes, config.CLASSES)):
    beats = X[y == i]
    if len(beats) == 0:
        ax.set_title(f"{c} (n=0)"); continue
    mu, sd = beats.mean(0), beats.std(0)
    ax.plot(t, mu); ax.fill_between(t, mu - sd, mu + sd, alpha=0.25)
    ax.set_title(f"{c} (n={len(beats):,})"); ax.set_xlabel("ms from R-peak")
axes[0].set_ylabel("z-scored amplitude")
plt.tight_layout(); plt.show()

## 3. Timing: premature beats
S and V beats usually arrive early, so `pre_RR / local average` drops below 1.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for c in ["N", "S", "V", "F"]:
    vals = df.loc[df.label == c, "pre_rr_ratio"].clip(0, 2)
    ax.hist(vals, bins=80, density=True, histtype="step", linewidth=1.5, label=c)
ax.axvline(1, color="grey", linestyle="--")
ax.set_xlabel("pre-RR / local average RR"); ax.set_ylabel("density"); ax.legend()
plt.show()

## 4. Patients differ a lot
The average normal beat looks different from patient to patient. If the same patient is in train and test, a model can memorise the patient instead of learning the arrhythmia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for p in ["100", "105", "119", "208", "232"]:
    sel = (patient == p) & (y == 0)
    if sel.any():
        ax.plot(t, X[sel].mean(0), label=f"patient {p}")
ax.set_xlabel("ms from R-peak"); ax.set_title("Average normal (N) beat by patient"); ax.legend()
plt.show()

## Notes
Write 2–3 takeaways here; they make good README and interview material.